# LGT Order Refine Train

`checkpoint-3500` LoRA adapter를 초기값으로 불러와 Adjacent 없이 `order / pairwise / first / last` 네 태스크만 추가 학습한다.

이 노트북은 학습만 수행한다. Validation 평가와 test submission은 `lgt_order_refine_eval_infer.ipynb`에서 따로 실행한다.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_lgt_order_refine_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"


def ensure_base_model_path():
    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR

    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id:", MODEL_REPO_ID)
        return MODEL_REPO_ID

    print("Base model cache not found. Downloading via ModelScope:")
    print(DRIVE_MODEL_DIR)
    from modelscope import snapshot_download as modelscope_snapshot_download

    model_dir = modelscope_snapshot_download(
        MODEL_REPO_ID,
        cache_dir="/content/modelscope_cache",
    )
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not os.path.exists(DRIVE_MODEL_DIR):
        shutil.copytree(model_dir, DRIVE_MODEL_DIR)
    print("Base model cached at:", DRIVE_MODEL_DIR)
    return DRIVE_MODEL_DIR


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
INITIAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_multitask_v1/runs/"
    "20260712_234828/lgt_multitask/checkpoint-3500"
)
OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "lgt_order_refine")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_lgt_order_refine.csv")

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None

TASK_RATIOS = {
    "order": 0.40,
    "pairwise": 0.20,
    "first": 0.20,
    "last": 0.20,
}
TASK_LOSS_WEIGHTS = {
    "order": 1.0,
    "pairwise": 1.0,
    "first": 1.0,
    "last": 1.0,
}

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
ALPHAS = [0.5, 1.0, 1.5, 2.0]
BETAS = [0.5, 1.0, 1.5, 2.0]
GAMMAS = [0.5, 1.0, 1.5, 2.0]

for path in [OUTPUT_ROOT, RUN_ROOT, OUTPUT_DIR, EVAL_DIR]:
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(INITIAL_ADAPTER_DIR, "adapter_config.json")), INITIAL_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("initial adapter:", INITIAL_ADAPTER_DIR)
print("output:", OUTPUT_DIR)

In [ ]:
# 3) Data split + multitask record generation
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def parse_order_prediction(text):
    match = re.fullmatch(r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*", str(text))
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def pairwise_target(answer, first_index, second_index):
    return "1" if int(answer[first_index]) < int(answer[second_index]) else "2"


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": format_order(base["order"])})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(base["order"][0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(base["order"][-1])})
        pools["last"].append(item)

        for first_index, second_index in PAIR_INDICES:
            item = copy.deepcopy(base)
            item.update({
                "task_type": "pairwise",
                "first_index": first_index,
                "second_index": second_index,
                "image_paths": [base["image_paths"][first_index], base["image_paths"][second_index]],
                "target": pairwise_target(base["answer"], first_index, second_index),
            })
            pools["pairwise"].append(item)
    return pools


def sample_records(records, count, rng):
    indices = rng.integers(0, len(records), size=count)
    return [records[int(index)] for index in indices]


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        records = sample_records(pools[task], count, rng)
        merged.extend(records)
        print(task, "pool:", len(pools[task]), "sampled:", len(records))
    rng.shuffle(merged)
    return merged, pools


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)

with open(os.path.join(RUN_ROOT, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump({
        "initial_adapter_dir": INITIAL_ADAPTER_DIR,
        "output_dir": OUTPUT_DIR,
        "task_ratios": TASK_RATIOS,
        "task_loss_weights": TASK_LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "max_train_steps": MAX_TRAIN_STEPS,
        "save_steps": SAVE_STEPS,
        "seed": SEED,
        "train_rows": len(training_df),
        "validation_rows": len(validation_df),
    }, f, ensure_ascii=False, indent=2)

print("train/valid:", len(training_df), len(validation_df))
print("train records:", len(train_records))

In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the beginning of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "last":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the end of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n"
            "Using these cues, determine the complete chronological order.\n"
            "Output only the final ordered list, such as [1, 2, 3, 4]."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class LGTRefineDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class LGTRefineCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            text = self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False)
            texts.append(text)
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["task_type"] = task_types
        return encoded

In [ ]:
# 5) Load checkpoint-3500 adapter and build trainer
class TaskLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        task_types = inputs.pop("task_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        weights = torch.tensor(
            [TASK_LOSS_WEIGHTS.get(task, 1.0) for task in task_types],
            dtype=sample_losses.dtype,
            device=sample_losses.device,
        )
        loss = (sample_losses * weights).mean()
        logs = {}
        for task in sorted(set(task_types)):
            task_mask = torch.tensor([value == task for value in task_types], device=sample_losses.device)
            if task_mask.any():
                logs[f"train_{task}_loss"] = sample_losses[task_mask].mean().detach().float().item()
        if logs:
            self.log(logs)
        return (loss, outputs) if return_outputs else loss


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(base_model, INITIAL_ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
print("trainable parameter module count:", len(trainable_names))
model.print_trainable_parameters()

training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_TRAIN_STEPS,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "logging_steps": LOGGING_STEPS,
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": None,
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "seed": SEED,
    "data_seed": SEED,
}
training_args = TrainingArguments(**training_argument_values)

trainer = TaskLossTrainer(
    model=model,
    args=training_args,
    train_dataset=LGTRefineDataset(train_records),
    data_collator=LGTRefineCollator(processor),
)

In [ ]:
# 6) Train only. No validation or inference is run in this notebook.
# max_steps = -1 means train by num_train_epochs.
trainer.train()
model.save_pretrained(os.path.join(OUTPUT_DIR, "final_adapter"))
processor.save_pretrained(OUTPUT_DIR)
print("saved:", OUTPUT_DIR)

## 다음 단계

학습이 끝나거나 중간 checkpoint가 충분히 쌓이면 `lgt_order_refine_eval_infer.ipynb`를 실행한다.

저장 위치:

```text
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/{RUN_ID}/lgt_order_refine/
```

100 step마다 `checkpoint-*`가 저장된다.